# Unconditional SE(3) Diffusion Training

Trains an **unconditional** SE(3) score network on a single protein MD
trajectory following the SinFusion single-trajectory protocol.

**What you get:** A model that generates new protein backbone conformations
from noise, sampling from the learned equilibrium distribution.

| Step | Description |
|------|-------------|
| 1 | Mount Google Drive (persistent storage) |
| 2 | Clone repo & init submodules |
| 3 | Configure protein & training |
| 4 | Download & preprocess trajectory |
| 5 | Train |
| 6 | Generate samples |

## Step 1: Environment Setup

In [ ]:
# Detect environment
try:
    import google.colab
    IN_COLAB = True
    print('Running in Google Colab')
except ImportError:
    IN_COLAB = False
    print('Running locally')

In [ ]:
import os

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/protein_data/data', exist_ok=True)
    print('Drive mounted')

## Step 2: Clone Repository & Init Submodules

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/JiwonJJeong/winter-gen-pproject.git'

if IN_COLAB:
    if not os.path.exists('winter-gen-pproject'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('winter-gen-pproject')
    subprocess.run(['git', 'pull', 'origin', 'main'], check=True)
    subprocess.run(['git', 'submodule', 'update', '--init', '--recursive'], check=True)
    # Symlink data/ to Drive for persistence
    if not os.path.islink('data'):
        os.symlink('/content/drive/MyDrive/protein_data/data', 'data')
        print('data/ -> /content/drive/MyDrive/protein_data/data')
    print('Code ready')
else:
    assert os.path.isdir('gen_model'), 'Run from repo root'
    print(f'Local repo: {os.path.abspath(".")}')

## Step 3: Configuration

Edit the variables below for your protein and training preferences.

In [ ]:
# ---- Protein (single-trajectory, SinFusion-style) ----
PROTEIN   = '4o66_C'   # Protein name (as in atlas.csv)
REPLICA   = '1'        # Which replica to train on (1, 2, or 3)

# ---- Generalization test (ill-posed but informative) ----
# Train on PROTEIN, evaluate on both PROTEIN and EVAL_PROTEIN
# to quantify single-trajectory overfitting vs. generalization.
EVAL_PROTEIN = '6ono_C'  # 76-residue protein from MDGen/STAR-MD test set (never in any training split)

# ---- Training ----
MAX_STEPS  = 50_000    # 50k for Colab; 200k for full training   # Total training steps
BATCH_SIZE = 1         # SinFusion default for single-trajectory
LR         = 1e-4
GRAD_CLIP  = 1.0       # Gradient clipping (MDGen default)
EMA_DECAY  = 0.999     # EMA decay (0 = disabled)

# ---- Paths ----
DATA_DIR = './data'
# On Colab, save to Drive so checkpoints persist across sessions
SAVE_DIR = f'/content/drive/MyDrive/protein_data/checkpoints/' if IN_COLAB else f'checkpoints/'
SAVE_DIR = SAVE_DIR + f'unconditional/{PROTEIN}'

print(f'Train protein : {PROTEIN}_R{REPLICA}')
print(f'Eval protein  : {EVAL_PROTEIN}')
print(f'Steps         : {MAX_STEPS:,}')
print(f'Save to       : {SAVE_DIR}')


## Step 4: Download & Prepare Data

In [ ]:
import os

if IN_COLAB and not os.path.islink('data'):
    raise RuntimeError('data/ not linked to Drive. Run Step 1 first.')

# Download training protein
!python scripts/download_and_prep.py {PROTEIN} --data_dir {DATA_DIR} --out_dir {DATA_DIR} --suffix _latent

# Download evaluation protein (for generalization test)
!python scripts/download_and_prep.py {EVAL_PROTEIN} --data_dir {DATA_DIR} --out_dir {DATA_DIR} --suffix _latent

for prot in [PROTEIN, EVAL_PROTEIN]:
    traj_path = f'{DATA_DIR}/{prot}/{prot}_R1_latent.npy'
    if os.path.exists(traj_path):
        import numpy as np
        arr = np.load(traj_path)
        print(f'  {prot}: {traj_path}  shape={arr.shape}')
    else:
        print(f'  ERROR: {traj_path} not found')


## Step 5: Train

Runs `gen_model/train_unconditional.py` with all features:
EMA, gradient clipping, cosine LR, mixed precision.

In [ ]:
!python gen_model/train_unconditional.py \
    --protein {PROTEIN} \
    --replica {REPLICA} \
    --data_dir {DATA_DIR} \
    --batch_size {BATCH_SIZE} \
    --max_steps {MAX_STEPS} \
    --lr {LR} \
    --grad_clip {GRAD_CLIP} \
    --ema_decay {EMA_DECAY} \
    --save_dir {SAVE_DIR} \
    --num_blocks 4 \
    --num_workers 2

## Step 6: Generate Samples (Training Protein)

In [ ]:
import glob

ckpts = sorted(glob.glob(f'{SAVE_DIR}/uncond-*.ckpt'))
best_ckpt = ckpts[-1] if ckpts else f'{SAVE_DIR}/last.ckpt'
print(f'Checkpoint: {best_ckpt}')

!python gen_model/inference_unconditional.py \
    --checkpoint {best_ckpt} \
    --npy_path {DATA_DIR}/{PROTEIN}/{PROTEIN}_R{REPLICA}_latent.npy \
    --atlas_csv gen_model/splits/atlas.csv \
    --protein_name {PROTEIN} \
    --num_samples 100 \
    --num_steps 200 \
    --out_dir outputs/unconditional/{PROTEIN}


## Step 7: Evaluate (Training Protein)

Compare generated samples against reference MD (all 3 replicas).

In [ ]:
!python gen_model/evaluate.py \
    --ref_npy {DATA_DIR}/{PROTEIN}/{PROTEIN}_R1_latent.npy \
              {DATA_DIR}/{PROTEIN}/{PROTEIN}_R2_latent.npy \
              {DATA_DIR}/{PROTEIN}/{PROTEIN}_R3_latent.npy \
    --gen_traj outputs/unconditional/{PROTEIN}/ca_samples.npy \
    --protein {PROTEIN} --mode unconditional --ref_mode all \
    --out_dir outputs/eval/unconditional/{PROTEIN}


## Step 8: Visualize (Training Protein)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ca_path = f'outputs/unconditional/{PROTEIN}/ca_samples.npy'
if os.path.exists(ca_path):
    samples = np.load(ca_path)
    print(f'Generated: {samples.shape}')
    fig, axes = plt.subplots(1, min(5, len(samples)), figsize=(15, 3))
    if len(samples) == 1: axes = [axes]
    for i, ax in enumerate(axes):
        ca = samples[i]
        ax.plot(ca[:, 0], ca[:, 1], '-', lw=0.5)
        ax.set_title(f'Sample {i+1}')
        ax.set_aspect('equal'); ax.axis('off')
    plt.suptitle(f'{PROTEIN} (training protein) — Generated CA traces')
    plt.tight_layout(); plt.show()
else:
    print('No samples found. Run inference first.')


## Step 9: Generalization Test

Generate and evaluate on `EVAL_PROTEIN` — a held-out protein from the
MDGen/STAR-MD test set that was never in any training split.
This quantifies how much the single-trajectory model overfits vs. generalizes.

In [ ]:
print(f'Generalization test: {PROTEIN} (train) -> {EVAL_PROTEIN} (held-out)')
print(f'  {EVAL_PROTEIN} is from the MDGen/STAR-MD test set')

# Generate on held-out protein
!python gen_model/inference_unconditional.py \
    --checkpoint {best_ckpt} \
    --npy_path {DATA_DIR}/{EVAL_PROTEIN}/{EVAL_PROTEIN}_R1_latent.npy \
    --atlas_csv gen_model/splits/atlas.csv \
    --protein_name {EVAL_PROTEIN} \
    --num_samples 100 \
    --num_steps 200 \
    --out_dir outputs/unconditional/{EVAL_PROTEIN}

# Evaluate against all 3 replicas
!python gen_model/evaluate.py \
    --ref_npy {DATA_DIR}/{EVAL_PROTEIN}/{EVAL_PROTEIN}_R1_latent.npy \
              {DATA_DIR}/{EVAL_PROTEIN}/{EVAL_PROTEIN}_R2_latent.npy \
              {DATA_DIR}/{EVAL_PROTEIN}/{EVAL_PROTEIN}_R3_latent.npy \
    --gen_traj outputs/unconditional/{EVAL_PROTEIN}/ca_samples.npy \
    --protein {EVAL_PROTEIN} --mode unconditional --ref_mode all \
    --out_dir outputs/eval/unconditional/{EVAL_PROTEIN}


In [ ]:
# Visualize generalization samples
import numpy as np
import matplotlib.pyplot as plt

ca_path = f'outputs/unconditional/{EVAL_PROTEIN}/ca_samples.npy'
if os.path.exists(ca_path):
    samples = np.load(ca_path)
    print(f'Generated: {samples.shape}')
    fig, axes = plt.subplots(1, min(5, len(samples)), figsize=(15, 3))
    if len(samples) == 1: axes = [axes]
    for i, ax in enumerate(axes):
        ca = samples[i]
        ax.plot(ca[:, 0], ca[:, 1], '-', lw=0.5)
        ax.set_title(f'Sample {i+1}')
        ax.set_aspect('equal'); ax.axis('off')
    plt.suptitle(f'{EVAL_PROTEIN} (held-out protein) — Generated CA traces')
    plt.tight_layout(); plt.show()
else:
    print('No samples found.')


## Step 10: MDGen Baseline

Download the pre-trained MDGen ATLAS checkpoint from HuggingFace and run
inference on both proteins. MDGen is trained on 1266 ATLAS proteins
(multi-trajectory), so this provides an **upper-bound baseline** for
comparison against our single-trajectory model.

Model weights: https://huggingface.co/bjing-mit/mdgen

In [ ]:
import os, subprocess

# Download MDGen ATLAS checkpoint from HuggingFace
MDGEN_CKPT_DIR = 'checkpoints/mdgen'
MDGEN_CKPT = os.path.join(MDGEN_CKPT_DIR, 'atlas.ckpt')

if not os.path.exists(MDGEN_CKPT):
    os.makedirs(MDGEN_CKPT_DIR, exist_ok=True)
    print('Downloading MDGen ATLAS checkpoint from HuggingFace ...')
    subprocess.run([
        'huggingface-cli', 'download', 'bjing-mit/mdgen', 'atlas.ckpt',
        '--local-dir', MDGEN_CKPT_DIR,
    ], check=True)
    print(f'Downloaded: {MDGEN_CKPT}')
else:
    print(f'MDGen checkpoint exists: {MDGEN_CKPT}')


In [ ]:
import sys, os

# MDGen inference script lives in the extern submodule
mdgen_script = 'extern/mdgen/sim_inference.py'
mdgen_out = 'outputs/mdgen_baseline'
os.makedirs(mdgen_out, exist_ok=True)

# Run MDGen on training protein
print(f'Running MDGen on {PROTEIN} ...')
!python {mdgen_script} \
    --sim_ckpt {MDGEN_CKPT} \
    --data_dir {DATA_DIR} \
    --split gen_model/splits/atlas.csv \
    --pdb_id {PROTEIN} \
    --num_frames 250 --num_rollouts 1 \
    --suffix _R1_latent \
    --xtc \
    --out_dir {mdgen_out}

# Run MDGen on eval protein
print(f'\nRunning MDGen on {EVAL_PROTEIN} ...')
!python {mdgen_script} \
    --sim_ckpt {MDGEN_CKPT} \
    --data_dir {DATA_DIR} \
    --split gen_model/splits/atlas.csv \
    --pdb_id {EVAL_PROTEIN} \
    --num_frames 250 --num_rollouts 1 \
    --suffix _R1_latent \
    --xtc \
    --out_dir {mdgen_out}


In [ ]:
# Evaluate MDGen baseline on both proteins
# MDGen outputs PDB/XTC directly, so we can run analyze_peptide_sim.py
# But for consistency, use our evaluate.py pipeline

for prot in [PROTEIN, EVAL_PROTEIN]:
    mdgen_pdb = f'{mdgen_out}/{prot}.pdb'
    if os.path.exists(mdgen_pdb):
        print(f'\n{"="*60}')
        print(f'MDGen baseline evaluation: {prot}')
        print(f'{"="*60}')
        # Run MDGen's own analysis script directly on its output
        ref_parent = f'outputs/eval/mdgen_baseline/{prot}/ref/{prot}'
        os.makedirs(ref_parent, exist_ok=True)
        # Write reference PDB/XTC
        !python -c "
import numpy as np, sys; sys.path.insert(0,'extern/mdgen')
from mdgen.utils import atom14_to_pdb
from mdgen.residue_constants import restype_order
import pandas as pd, mdtraj
seq_df = pd.read_csv('gen_model/splits/atlas.csv', index_col='name')
seqres = seq_df.loc['{prot}', 'seqres']
aatype = np.array([restype_order[c] for c in seqres])
refs = [np.load(f'{DATA_DIR}/{prot}/{prot}_R{{r}}_latent.npy').astype(np.float32) for r in [1,2,3]]
ref = np.concatenate(refs)
atom14_to_pdb(ref, aatype, '{ref_parent}/{prot}.pdb')
t = mdtraj.load('{ref_parent}/{prot}.pdb'); t.superpose(t)
t[0].save('{ref_parent}/{prot}.pdb'); t.save('{ref_parent}/{prot}.xtc')
"
        # Run MDGen analysis
        !python extern/mdgen/scripts/analyze_peptide_sim.py \
            --mddir outputs/eval/mdgen_baseline/{prot}/ref \
            --pdbdir {mdgen_out} \
            --pdb_id {prot} \
            --save --save_name mdgen_eval.pkl \
            --no_msm --plot
    else:
        print(f'MDGen output not found for {prot}')


## Step 11: Naive Single-Trajectory MDGen Baseline

Train MDGen's own architecture on the **same single trajectory** as our
model (no SinFusion anti-overfitting, no STAR-MD attention). This isolates
the contribution of our architectural changes vs. naive single-trajectory
training with MDGen's flow matching approach.

Comparison:
- **Our model** (Steps 6-8): STAR-MD + SinFusion protocol
- **MDGen pre-trained** (Step 10): multi-trajectory upper bound
- **MDGen naive** (this step): same data, MDGen architecture, no anti-overfitting

In [ ]:
import os, subprocess

# Create a single-protein split CSV for MDGen training
import pandas as pd

atlas_df = pd.read_csv('gen_model/splits/atlas.csv', index_col='name')
single_train = atlas_df.loc[[PROTEIN]]
single_val = atlas_df.loc[[PROTEIN]]  # overfit check: val = train

os.makedirs('outputs/mdgen_naive', exist_ok=True)
train_split = f'outputs/mdgen_naive/{PROTEIN}_train.csv'
val_split   = f'outputs/mdgen_naive/{PROTEIN}_val.csv'
single_train.to_csv(train_split)
single_val.to_csv(val_split)
print(f'Created single-protein splits: {train_split}')


In [ ]:
import os

MDGEN_NAIVE_DIR = f'checkpoints/mdgen_naive/{PROTEIN}'
os.makedirs(MDGEN_NAIVE_DIR, exist_ok=True)
os.environ['MODEL_DIR'] = MDGEN_NAIVE_DIR

# Train MDGen on single protein with its ATLAS config
# (num_frames=250, batch_size=1, prepend_ipa, crop up to 256 residues)
# Reduced epochs since we're training on ~1/1000th of the data
MDGEN_EPOCHS = 100     # 100 for Colab; 500 for full training

!python extern/mdgen/train.py \
    --sim_condition \
    --train_split {train_split} \
    --val_split {val_split} \
    --data_dir {DATA_DIR} \
    --num_frames 250 \
    --batch_size 1 \
    --prepend_ipa \
    --crop 256 \
    --val_repeat 1 \
    --epochs {MDGEN_EPOCHS} \
    --atlas \
    --ckpt_freq 50 \
    --suffix _R{REPLICA}_latent \
    --no_validate \
    --run_name mdgen_naive_{PROTEIN}


In [ ]:
import glob

# Find latest checkpoint
naive_ckpts = sorted(glob.glob(f'{MDGEN_NAIVE_DIR}/*.ckpt'))
naive_ckpt = naive_ckpts[-1] if naive_ckpts else None

if naive_ckpt:
    print(f'Naive MDGen checkpoint: {naive_ckpt}')
    mdgen_naive_out = 'outputs/mdgen_naive/inference'
    os.makedirs(mdgen_naive_out, exist_ok=True)

    # Run on training protein
    !python extern/mdgen/sim_inference.py \
        --sim_ckpt {naive_ckpt} \
        --data_dir {DATA_DIR} \
        --split gen_model/splits/atlas.csv \
        --pdb_id {PROTEIN} \
        --num_frames 250 --num_rollouts 1 \
        --suffix _R1_latent \
        --xtc \
        --out_dir {mdgen_naive_out}

    # Run on eval protein
    !python extern/mdgen/sim_inference.py \
        --sim_ckpt {naive_ckpt} \
        --data_dir {DATA_DIR} \
        --split gen_model/splits/atlas.csv \
        --pdb_id {EVAL_PROTEIN} \
        --num_frames 250 --num_rollouts 1 \
        --suffix _R1_latent \
        --xtc \
        --out_dir {mdgen_naive_out}

    # Evaluate
    for prot in [PROTEIN, EVAL_PROTEIN]:
        pdb = f'{mdgen_naive_out}/{prot}.pdb'
        if os.path.exists(pdb):
            print(f'\nEvaluating naive MDGen on {prot} ...')
            ref_dir = f'outputs/eval/mdgen_naive/{prot}/ref/{prot}'
            os.makedirs(ref_dir, exist_ok=True)
            !python -c "
import numpy as np, sys; sys.path.insert(0,'extern/mdgen')
from mdgen.utils import atom14_to_pdb
from mdgen.residue_constants import restype_order
import pandas as pd, mdtraj
seq_df = pd.read_csv('gen_model/splits/atlas.csv', index_col='name')
seqres = seq_df.loc['{prot}', 'seqres']
aatype = np.array([restype_order[c] for c in seqres])
refs = [np.load(f'{DATA_DIR}/{prot}/{prot}_R{{r}}_latent.npy').astype(np.float32) for r in [1,2,3]]
ref = np.concatenate(refs)
atom14_to_pdb(ref, aatype, '{ref_dir}/{prot}.pdb')
t = mdtraj.load('{ref_dir}/{prot}.pdb'); t.superpose(t)
t[0].save('{ref_dir}/{prot}.pdb'); t.save('{ref_dir}/{prot}.xtc')
"
            !python extern/mdgen/scripts/analyze_peptide_sim.py \
                --mddir outputs/eval/mdgen_naive/{prot}/ref \
                --pdbdir {mdgen_naive_out} \
                --pdb_id {prot} \
                --save --save_name mdgen_naive_eval.pkl \
                --no_msm --plot
else:
    print('No naive MDGen checkpoint found. Training may have failed.')
